# Q3: Trees and Random Forests

In [1]:
# Imports necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import os
import shutil
import math
import nbformat
from collections import Counter
from typing import List, Tuple, Dict, Optional, Union, Any

# Scikit-learn imports
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import balanced_accuracy_score, make_scorer

# 1. Setup Artifacts Directory
artifacts_dir = 'runs-q3'
if os.path.exists(artifacts_dir):
    print(f"Removing old artifacts directory: {artifacts_dir}")
    shutil.rmtree(artifacts_dir)
print(f"Creating clean artifacts directory: {artifacts_dir}")
os.makedirs(artifacts_dir)

# 2. Fix seed for reproducible randomization.
username = "sudhanva.joshi"
seed = int(hashlib.sha256(username.encode()).hexdigest(), 16) % (2**32)
print("Seed generated:", seed)
rng = np.random.default_rng(seed)
np.random.seed(seed) 

# 3. Set plotting theme and helper functons.
sns.set_theme(context="paper", style="darkgrid",
              palette="magma", font="Nimbus Roman")

def plotinfo(title, x, y, userid=username, filename=None):
    """Helper function for plotting."""
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel(x, fontsize=12)
    plt.ylabel(y, fontsize=12)
    plt.text(0.98, 0.02, userid, ha="right", va="bottom",
             transform=plt.gca().transAxes, fontsize=10,
             color="gray", alpha=0.7)
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(artifacts_dir, filename))
    plt.show()

print("Setup complete.")

Removing old artifacts directory: runs-q3
Creating clean artifacts directory: runs-q3
Seed generated: 1027714457
Setup complete.


## Dataset: Bag-of-words for Amazon Product Reviews

In [ ]:
# 4. Data Loading (Adapted for CSV files)
def load_data_from_csv(split_name: str, base_path: str = './data/data_product_reviews/'):
    """Loads a single split of the BoW data from CSV files."""
    x_path = os.path.join(base_path, f'x_{split_name}.csv')
    y_path = os.path.join(base_path, f'y_{split_name}.csv')
    
    print(f"Loading {split_name} data from {x_path} and {y_path}.")
    
    x_df = pd.read_csv(x_path)
    y_df = pd.read_csv(y_path)
    
    X = x_df.values
    y = y_df.values.ravel()  # Flatten to 1D array
    
    vocab = x_df.columns.tolist() if split_name == 'train' else []
    
    print(f"Loaded {split_name} data: X shape {X.shape}, y shape {y.shape}")
    return X, y, vocab

X_train, y_train, vocab = load_data_from_csv('train')
X_val, y_val, _ = load_data_from_csv('valid')
X_test, y_test, _ = load_data_from_csv('test')

# Verify vocab size matches feature dimension
assert len(vocab) == X_train.shape[1], "Vocabulary size mismatch with feature dimension."

print(f"\nData loading successful. Vocabulary size: {len(vocab)}")

# Combine train and val sets
X_train_val = np.vstack((X_train, X_val))
y_train_val = np.concatenate((y_train, y_val))

# Create 'test_fold' array: -1 for train, 0 for val
train_indices = np.full(X_train.shape[0], -1, dtype=int)
val_indices = np.zeros(X_val.shape[0], dtype=int)
test_fold = np.concatenate((train_indices, val_indices))

my_splitter = PredefinedSplit(test_fold)

print(f"\nCreated combined train/val set: X shape {X_train_val.shape}")
print(f"Created PredefinedSplit for GridSearchCV.")

Loading train data from ./Q3/data_product_reviews/x_train.csv and ./Q3/data_product_reviews/y_train.csv.
Loaded train data: X shape (6346, 7729), y shape (6346,)
Loading valid data from ./Q3/data_product_reviews/x_valid.csv and ./Q3/data_product_reviews/y_valid.csv.
Loaded valid data: X shape (792, 7729), y shape (792,)
Loading test data from ./Q3/data_product_reviews/x_test.csv and ./Q3/data_product_reviews/y_test.csv.
Loaded test data: X shape (793, 7729), y shape (793,)

Data loading successful. Vocabulary size: 7729

Created combined train/val set: X shape (7138, 7729)
Created PredefinedSplit for GridSearchCV.


## Decision Tree Implementation from Scratch

In [ ]:
# Part 1A & 1B: Node Implementation

class LeafNode:
    """A leaf node in a decision tree. Holds a single class prediction."""
    def __init__(self, prediction: int):
        self.prediction = prediction

    def predict(self, x: np.ndarray) -> int:
        """Predicts the class for a single data point."""
        return self.prediction

class InternalDecisionNode:
    """An internal node in a decision tree. Splits data on a feature."""
    def __init__(self, feature_index: int, 
                 left_child: Union['InternalDecisionNode', LeafNode], 
                 right_child: Union['InternalDecisionNode', LeafNode]):
        self.feature_index = feature_index
        self.left_child = left_child
        self.right_child = right_child

    def predict(self, x: np.ndarray) -> int:
        """Predicts the class for a single data point by routing."""
        # For binary BoW data, 0 = feature absent (left), 1 = feature present (right)
        if x[self.feature_index] == 0:
            return self.left_child.predict(x)
        else:
            return self.right_child.predict(x)

print("\nTree node classes defined.")

# Part 1C: Best Split Selection

def entropy(y: np.ndarray) -> float:
    """Calculates the entropy of a set of labels."""
    if len(y) == 0:
        return 0.0
    
    p1 = np.mean(y) # Probability of class 1
    if p1 == 0 or p1 == 1:
        return 0.0
    
    p0 = 1 - p1
    return -p0 * math.log2(p0) - p1 * math.log2(p1)

def information_gain(y_parent: np.ndarray, y_left: np.ndarray, y_right: np.ndarray) -> float:
    """Calculates the information gain of a split."""
    n_parent = len(y_parent)
    if n_parent == 0:
        return 0.0
        
    n_left = len(y_left)
    n_right = len(y_right)
    
    h_parent = entropy(y_parent)
    h_left = entropy(y_left)
    h_right = entropy(y_right)
    
    weighted_avg_child_entropy = (n_left / n_parent) * h_left + (n_right / n_parent) * h_right
    
    return h_parent - weighted_avg_child_entropy

def select_best_binary_split(X: np.ndarray, y: np.ndarray) -> Optional[int]:
    """
    Finds the best feature to split on for binary data.
    Returns the feature index or None if no split improves gain.
    """
    best_feature_index = None
    best_info_gain = -1.0
    n_features = X.shape[1]

    for j in range(n_features):
        # Split data based on feature j (0 = absent, 1 = present)
        left_indices = (X[:, j] == 0)
        right_indices = (X[:, j] == 1)
        
        y_left = y[left_indices]
        y_right = y[right_indices]
        
        # Don't split if it results in an empty child
        if len(y_left) == 0 or len(y_right) == 0:
            continue
            
        gain = information_gain(y, y_left, y_right)
        
        if gain > best_info_gain:
            best_info_gain = gain
            best_feature_index = j
    
    # Only return a split if it's better than not splitting (gain > 0)
    if best_info_gain > 0:
        return best_feature_index
    else:
        return None

print("Split selection functions defined.")

# Part 1D: Tree Training

TreeNode = Union[InternalDecisionNode, LeafNode]

def get_majority_class(y: np.ndarray) -> int:
    """Returns the majority class (0 or 1)."""
    if len(y) == 0:
        return 1  # Default prediction if node is empty
    return Counter(y).most_common(1)[0][0]

def train_tree(
    X: np.ndarray, 
    y: np.ndarray, 
    max_depth: int, 
    min_samples_leaf: int, 
    current_depth: int = 0
) -> TreeNode:
    """Recursively trains a decision tree."""
    
    # 1. Base case: Check stopping conditions
    majority_class = get_majority_class(y)
    
    # Stop if node is pure
    if len(np.unique(y)) <= 1:
        return LeafNode(majority_class)
    
    # Stop if max depth reached
    if current_depth >= max_depth:
        return LeafNode(majority_class)
        
    # Stop if not enough samples
    if len(y) < min_samples_leaf:
        return LeafNode(majority_class)

    # 2. Recursive step: Find best split
    best_feature_index = select_best_binary_split(X, y)
    
    # Stop if no split provides information gain
    if best_feature_index is None:
        return LeafNode(majority_class)
    
    # 3. Split the data
    left_indices = (X[:, best_feature_index] == 0)
    right_indices = (X[:, best_feature_index] == 1)
    
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]

    # Stop if a split results in an empty child
    if len(y_left) == 0 or len(y_right) == 0:
        return LeafNode(majority_class)

    # 4. Build children
    left_child = train_tree(
        X_left, y_left, max_depth, min_samples_leaf, current_depth + 1
    )
    right_child = train_tree(
        X_right, y_right, max_depth, min_samples_leaf, current_depth + 1
    )
    
    return InternalDecisionNode(best_feature_index, left_child, right_child)

print("Tree training function defined.")

def predict_batch(tree: TreeNode, X: np.ndarray) -> np.ndarray:
    """Makes predictions for a batch of data."""
    return np.array([tree.predict(x) for x in X])

print("\nTraining from-scratch Decision Tree.")
# Use shallow depth and min samples for speed
my_tree = train_tree(X_train, y_train, max_depth=5, min_samples_leaf=10)
print("Training complete.")

# Evaluate on validation set
y_pred_val = predict_batch(my_tree, X_val)
val_accuracy = balanced_accuracy_score(y_val, y_pred_val)

print(f"From-scratch Tree (max_depth=5) Validation Balanced Accuracy: {val_accuracy:.4f}")


Tree node classes defined.
Split selection functions defined.
Tree training function defined.

Training from-scratch Decision Tree.
Training complete.
From-scratch Tree (max_depth=5) Validation Balanced Accuracy: 0.5958


## Decision Trees for Review Classification

In the generated tree, the node excel <= 0.500 has two children. The first child, disappoint <= 0.500, predicts class: 0.0. The second child, disappoint > 0.500, also predicts class: 0.0.

This makes sense because the decision tree's goal at each step is to find the split that maximizes information gain (or minimizes Gini impurity), even if that split isn't perfect. 

In this case, the data at the excel <= 0.500 node was likely not "pure" (it had a mix of positive and negative reviews). When the algorithm split this data based on the word "disappoint," it found that both resulting groups (reviews with "disappoint" and reviews without it) still had a majority of negative (0.0) reviews.

In [4]:
def pretty_print_sklearn(tree_model, feature_names):
    """Helper function to print sklearn tree text."""
    text_representation = export_text(
        tree_model, 
        feature_names=feature_names, 
        spacing=3,
        decimals=3
    )
    print(text_representation)

print("Training simple DecisionTreeClassifier.")
# 1. Train the simple tree 
simple_tree = DecisionTreeClassifier(
    criterion='gini', 
    max_depth=3, 
    min_samples_leaf=1, 
    min_samples_split=2, 
    random_state=101
)
simple_tree.fit(X_train, y_train)

print("Training complete.")
print("\n ASCII Representation of Simple Tree (max_depth=3)")

# 2. Show the text representation
pretty_print_sklearn(simple_tree, vocab)

Training simple DecisionTreeClassifier.
Training complete.

 ASCII Representation of Simple Tree (max_depth=3)
|--- great <= 0.500
|   |--- excel <= 0.500
|   |   |--- disappoint <= 0.500
|   |   |   |--- class: 0.0
|   |   |--- disappoint >  0.500
|   |   |   |--- class: 0.0
|   |--- excel >  0.500
|   |   |--- disappoint <= 0.500
|   |   |   |--- class: 1.0
|   |   |--- disappoint >  0.500
|   |   |   |--- class: 0.0
|--- great >  0.500
|   |--- return <= 0.500
|   |   |--- bad <= 0.500
|   |   |   |--- class: 1.0
|   |   |--- bad >  0.500
|   |   |   |--- class: 0.0
|   |--- return >  0.500
|   |   |--- movie <= 0.500
|   |   |   |--- class: 0.0
|   |   |--- movie >  0.500
|   |   |   |--- class: 1.0



### Hyperparameter Tuning

The best values for maximum depth and minimum samples for the best tree is given in the output below.

In [5]:
print("\nStarting GridSearchCV for DecisionTreeClassifier.")

# 1. Define parameter grid
dt_param_grid = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9],
    'random_state': [101],
    'criterion': ['gini']
}

# 2. Initialize GridSearchCV
dt_grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=dt_param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter,  # Use the predefined train/val split
    return_train_score=True,
    refit=False,       # We don't need to refit, just find the best params
    n_jobs=-1          # Use all available cores
)

# 3. Run the grid search
dt_grid_search.fit(X_train_val, y_train_val)

print("GridSearchCV complete.")

# 4. Store results
dt_results_df = pd.DataFrame(dt_grid_search.cv_results_)
dt_results_df = dt_results_df.sort_values(by='rank_test_score')

print(f"\nBest parameters found: {dt_grid_search.best_params_}")
print(f"Best validation balanced_accuracy: {dt_grid_search.best_score_:.4f}")

print("\nTop 5 DT Grid Search Results")
print(dt_results_df[['param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'mean_train_score']].head())


Starting GridSearchCV for DecisionTreeClassifier.
GridSearchCV complete.

Best parameters found: {'criterion': 'gini', 'max_depth': 32, 'min_samples_leaf': 3, 'random_state': 101}
Best validation balanced_accuracy: 0.7324

Top 5 DT Grid Search Results
    param_max_depth  param_min_samples_leaf  mean_test_score  mean_train_score
7                32                       3         0.732380          0.877218
9               128                       1         0.730313          0.996216
6                32                       1         0.727378          0.918197
11              128                       9         0.723193          0.832337
10              128                       3         0.716495          0.924198


## Random Forests for Review Classification

In [6]:
print("\nPart 3: Scikit-Learn Random Forests")

print("Training simple RandomForestClassifier.")

# 1. Train the simple RF
simple_rf = RandomForestClassifier(
    max_depth=3,
    n_estimators=100,
    random_state=101,
    n_jobs=-1
)
simple_rf.fit(X_train, y_train)
print("Training complete.")

# 2. Get feature importances
importances = simple_rf.feature_importances_
vocab_np = np.array(vocab)

# 3. Get top 10 most important features
top_10_indices = np.argsort(importances)[-10:][::-1]
top_10_words = vocab_np[top_10_indices]
top_10_scores = importances[top_10_indices]

top_features_df = pd.DataFrame({
    'Top 10 Important Word': top_10_words,
    'Importance Score': top_10_scores
}).reset_index(drop=True)

# 4. Get 10 random near-zero importance features
near_zero_indices = np.where(importances < 0.00001)[0]
num_near_zero = len(near_zero_indices)

if num_near_zero >= 10:
    random_zero_indices = rng.choice(near_zero_indices, 10, replace=False)
    random_zero_words = vocab_np[random_zero_indices]
    random_zero_scores = importances[random_zero_indices]
else:
    random_zero_words = vocab_np[near_zero_indices]
    random_zero_scores = importances[near_zero_indices]
    pad_width = 10 - len(random_zero_words)
    random_zero_words = np.pad(random_zero_words, (0, pad_width), 'constant', constant_values=None)
    random_zero_scores = np.pad(random_zero_scores, (0, pad_width), 'constant', constant_values=None)

zero_features_df = pd.DataFrame({
    'Random Near-Zero Word': random_zero_words,
    'Importance Score': random_zero_scores
}).reset_index(drop=True)

# 5. Combine into one table
comparison_df = pd.concat([top_features_df, zero_features_df], axis=1)

print(f"\nTotal features with near-zero importance (< 0.00001): {num_near_zero}")
print("\nFeature Importance Comparison")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(comparison_df)


Part 3: Scikit-Learn Random Forests
Training simple RandomForestClassifier.
Training complete.

Total features with near-zero importance (< 0.00001): 7296

Feature Importance Comparison
  Top 10 Important Word  Importance Score Random Near-Zero Word  Importance Score
0                return          0.032990             to_follow               0.0
1                 excel          0.029485                 games               0.0
2                 great          0.028984               <num>_i               0.0
3                 worst          0.028407           another_one               0.0
4                  poor          0.026748               academi               0.0
5            disappoint          0.024952                 start               0.0
6            your_money          0.018002                 plugs               0.0
7                i_love          0.017955                 by_my               0.0
8              the_best          0.017662                  town            

## Hyperparameter Tuning

The `n_estimators` hyperparameter (the number of trees) primarily controls the trade-off between computational cost and performance. As you add more trees, the model's prediction gets more stable and accurate because you are averaging the "votes" of more independent trees.

Unlike `max_depth`, you generally cannot overfit by setting n_estimators to be too large. As you add more and more trees, the validation error will typically decrease and then plateau, but it won't start to increase again (which is the classic sign of overfitting). 

Each tree is trained on a different sample of data, so adding more trees just makes the average prediction more stable. The only downside of a very large `n_estimators` is that it takes more time and memory to train and make predictions.

In [7]:
print("\nStarting GridSearchCV for RandomForestClassifier.")

# 1. Define parameter grid
rf_param_grid = {
    'max_features': [3, 10, 33, 100, 333],
    'max_depth': [16, 32],
    'min_samples_leaf': [1],
    'n_estimators': [100],
    'random_state': [101]
}

# 2. Initialize GridSearchCV
rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=rf_param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter,  # Use the same predefined train/val split
    return_train_score=True,
    refit=False,
    n_jobs=-1
)

# 3. Run the grid search
rf_grid_search.fit(X_train_val, y_train_val)

print("GridSearchCV complete.")

# 4. Store results
rf_results_df = pd.DataFrame(rf_grid_search.cv_results_)
rf_results_df = rf_results_df.sort_values(by='rank_test_score')

best_rf_params = rf_grid_search.best_params_
print(f"\nBest parameters found: {best_rf_params}")
print(f"Best validation balanced_accuracy: {rf_grid_search.best_score_:.4f}")

print("\nTop 5 RF Grid Search Results")
print(rf_results_df[['param_max_features', 'param_max_depth', 'mean_test_score', 'mean_train_score']].head())


Starting GridSearchCV for RandomForestClassifier.
GridSearchCV complete.

Best parameters found: {'max_depth': 32, 'max_features': 33, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
Best validation balanced_accuracy: 0.8512

Top 5 RF Grid Search Results
   param_max_features  param_max_depth  mean_test_score  mean_train_score
7                  33               32         0.851204          0.964402
6                  10               32         0.843626          0.970070
2                  33               16         0.842439          0.923600
1                  10               16         0.840997          0.925010
3                 100               16         0.839402          0.920745


## Final Model Comparison


For both the single Decision Tree and the Random Forest, the tuned models ("Best DT" and "Best RF") dramatically outperformed their simple, shallow counterparts ("Simple DT" and "Simple RF"). This shows that the max_depth=3 models were too simple (they were underfitting) and that tuning max_depth and min_samples_leaf is critical.

The "Best RF" (Validation Accuracy: 0.8512) was significantly better than the "Best DT" (Validation Accuracy: 0.7324). Hence, the ensemble of many decorrelated trees (Random Forest) generalizes much better to new data than any single, optimized tree.

The final test set scores confirm this. The "Best RF" (Test Accuracy: 0.8369) is the clear winner, proving it is the most robust and accurate model for this sentiment analysis task.

In [8]:
print("\nPart 3.3: Final Model Comparison")

# We need to train the 'Best' models using the best params found
print("Training Best DT with optimal parameters.")
best_dt_params = dt_grid_search.best_params_
best_tree = DecisionTreeClassifier(**best_dt_params)
best_tree.fit(X_train, y_train)
print("Training complete.")

print("Training Best RF with optimal parameters.")
best_rf_params = rf_grid_search.best_params_
best_rf = RandomForestClassifier(**best_rf_params, n_jobs=-1)
best_rf.fit(X_train, y_train)
print("Training complete.")

# 1. Collate models
models = {
    "Simple DT": simple_tree,
    "Best DT": best_tree,
    "Simple RF": simple_rf,
    "Best RF": best_rf
}

# 2. Collate model metadata 
model_info = {
    "Simple DT": {"Num Trees": 1, "Max Depth": simple_tree.get_params()['max_depth']},
    "Best DT": {"Num Trees": 1, "Max Depth": best_tree.get_params()['max_depth']},
    "Simple RF": {"Num Trees": simple_rf.get_params()['n_estimators'], "Max Depth": simple_rf.get_params()['max_depth']},
    "Best RF": {"Num Trees": best_rf.get_params()['n_estimators'], "Max Depth": best_rf.get_params()['max_depth']}
}

# 3. Calculate balanced accuracy for all models on all splits
results = []
for name, model in models.items():
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    acc_train = balanced_accuracy_score(y_train, y_pred_train)
    acc_val = balanced_accuracy_score(y_val, y_pred_val)
    acc_test = balanced_accuracy_score(y_test, y_pred_test)
    
    results.append({
        "Model Type": name,
        "Num Trees": model_info[name]["Num Trees"],
        "Max Depth": model_info[name]["Max Depth"],
        "Train Accuracy": acc_train,
        "Validation Accuracy": acc_val,
        "Test Accuracy": acc_test
    })

# 4. Create final DataFrame
final_results_df = pd.DataFrame(results)
final_results_df = final_results_df.set_index("Model Type")

print("\n Final Model Comparison (Balanced Accuracy)")
print(final_results_df.to_string(float_format="%.4f"))


Part 3.3: Final Model Comparison
Training Best DT with optimal parameters.
Training complete.
Training Best RF with optimal parameters.
Training complete.

 Final Model Comparison (Balanced Accuracy)
            Num Trees  Max Depth  Train Accuracy  Validation Accuracy  Test Accuracy
Model Type                                                                          
Simple DT           1          3          0.6458               0.6446         0.6458
Best DT             1         32          0.8772               0.7324         0.7489
Simple RF         100          3          0.8191               0.7972         0.7782
Best RF           100         32          0.9644               0.8512         0.8369
